# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriKeerthi16-b/Machine-Learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
# Install DuckDB
!pip -q install duckdb

In [5]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully")

DuckDB connected successfully


In [9]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Enter your Hugging Face READ token (hf_...): ")

hf_token = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("SET VARIABLE hf_token = ?", [hf_token])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token'),
    SCOPE 'huggingface.co'
)
""")

print("Hugging Face + DuckDB connected")

KeyboardInterrupt: Interrupted by user

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the tables available in the warehouse
tables = con.execute("SHOW TABLES").fetchdf()

display(tables)

,name




**Unit of analysis:** One row represents one webpage/content item (`content_id`).

**Time window:** I use the observation window available in the warehouse snapshot. The analysis uses historical measurements available within that window and does not use future information.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
print(df.columns.tolist())

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check that the fields we plan to use exist

planned_fields = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
]

missing_fields = [c for c in planned_fields if c not in df.columns]

print("Missing planned fields:", missing_fields)

display(df[planned_fields].head())



#### Features
These are fields that can be known at the decision moment and may be used as model inputs:

- `impressions_90d` — recent search visibility
- `sessions_90d` — recent traffic
- `content_age_days` — age of the content
- `days_since_last_update` — content freshness
- `engagement_rate` — observed user engagement

#### Label
- `trend_direction` — observed directional outcome.
- `is_declining_label` = `trend_direction == "down"`

The label is a proxy for observed decline, not a future outcome.

#### Context
- `content_id` — identifies the content item
- `client_id` — identifies the client grouping
- date/window fields — identify the observation period

These fields help describe or group the observations but are not treated as predictive features.

#### Excluded
- `trend_direction` is excluded from the feature set because it is used to construct the label.
- Any field calculated from the label is excluded because it would leak the answer into the model.
- Any future-window measurement is excluded because it would not be available at the decision moment.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Grain check

I claim that one row represents one `content_id`. I verify this by comparing total rows with distinct content IDs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
total_rows = len(df)
unique_content_ids = df["content_id"].nunique()

print("Total rows:", total_rows)
print("Unique content_id values:", unique_content_ids)
print("Duplicate content_id rows:", total_rows - unique_content_ids)

### Count check

I report the number of observations and the number of unique content items in the analysis slice.

In [ ]:
print("Total observations:", len(df))
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

### Missing-value check

I check missingness in the fields used by the contract. Missing values are reported rather than silently treated as valid observations.

In [ ]:
check_fields = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
]

missing_report = (
    df[check_fields]
    .isna()
    .sum()
    .to_frame("missing_count")
)

missing_report["missing_pct"] = (
    missing_report["missing_count"] / len(df) * 100
).round(2)

display(missing_report)

### Time-window check

I verify the earliest and latest observation dates and report the number of distinct dates represented in the analysis slice.

In [ ]:
print(df.columns.tolist())

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




### Unequal history
Not every content item necessarily has the same amount of historical information. Comparisons between items should therefore be interpreted as measured observations within the available history.

### GSC-only early rows
Some early observations may have more limited Google Search Console coverage. Metrics from those periods may therefore not be directly comparable with periods containing broader data coverage.

### Window overlap
Rolling windows such as 90-day measurements can overlap between nearby observations. Therefore, adjacent observations can share much of the same underlying history and should not automatically be treated as independent observations.

### Label limitation
`is_declining_label` is based on the observed `trend_direction` in the current data window. It is a proxy for observed decline and does not prove that a page will decline in a future period.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.